# DATA209 — Advanced Exploratory Data Analysis## Lab Practical P1–2 · Summary Statistics, Distribution Shape and Pivot Tables**Programme** B.Tech (Hons.) Data Science, Semester III · **School of Engineering and Technology, Vidyashilp University****Dataset** Online Shoppers Purchasing Intention (UCI Machine Learning Repository) — 12,330 sessions × 18 columns---### What you will do in this practical| # | Step | Skill being built ||---|------|-------------------|| 1 | State the problem precisely | Framing an EDA question that has an answer || 2 | Load the data and verify it | Never trusting a file you have not checked || 3 | Separate dependent from independent variables | Knowing what you are predicting and from what || 4 | Mean vs. median | Detecting asymmetry from two numbers || 5 | Variance vs. IQR | Choosing a spread measure that survives outliers || 6 | Skewness and shape | Naming the shape and deciding what to do about it || 7 | Histogram, box plot, density | Seeing what the summary hides || 8 | Pivot tables | Turning a raw table into a comparison || 9 | Findings log | Writing down what you learned, in one page |### How to use this notebookRun the cells in order. Every computation is followed by an **Interpretation** cell that reads the numbersyou just produced and states what they mean — read those carefully, because the exercises at the end askyou to write your own. Nothing in this notebook is hard-coded from a previous run: every printedinterpretation is generated from the data in front of you.

---## 1. Problem statement### 1.1 The business situationAn online retailer records one row for every visit to its website. Some visits end in a purchase; theoverwhelming majority do not. The marketing team wants to know **which sessions are worth intervening in** —a live-chat prompt, a discount banner, a retargeting cookie — because each intervention costs money andannoys the customer if it is misdirected.Before anybody builds a model, somebody has to answer a more basic question.### 1.2 The analytical question for P1–2> **How is browsing behaviour distributed across sessions, and how does it differ between sessions that end> in a purchase and sessions that do not?**This is deliberately *not* "build a classifier". P1–2 is a profiling exercise. Its output is a set ofdescriptive claims, each backed by a number and a plot, which the later practicals will build on.Three sub-questions make it concrete:1. **Shape.** Are the behavioural columns symmetric, or skewed? Which summary statistic honestly describes   a typical session?2. **Spread.** How much do sessions vary, and is that variation driven by the bulk of the data or by a   handful of extreme visits?3. **Contrast.** On which variables do purchasing and non-purchasing sessions actually differ, and by how much?### 1.3 Why this ordering mattersIf you answer question 3 before questions 1 and 2, you will compare the *mean* of a heavily skewed variableacross two groups, and you will get a difference that is driven by a few extreme sessions rather than bytypical behaviour. That mistake is invisible in the output and fatal in the conclusion. The whole point ofP1–2 is to make it impossible.

---## 2. SetupWe use only the standard scientific Python stack. `scipy.stats` supplies skewness and kurtosis;everything else is pandas, NumPy, Matplotlib and seaborn.

In [ ]:
import warningswarnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import stats# Display settings: wide enough to read, precise enough to comparepd.set_option("display.max_columns", 50)pd.set_option("display.width", 160)pd.set_option("display.float_format", lambda v: f"{v:,.3f}")# A consistent, readable plot style for the whole notebooksns.set_theme(style="whitegrid", context="notebook")plt.rcParams["figure.dpi"] = 110plt.rcParams["axes.titleweight"] = "bold"plt.rcParams["axes.titlesize"] = 12plt.rcParams["figure.autolayout"] = TruePRIMARY, SECONDARY, MUTED = "#B85042", "#6E8B7B", "#8C8073"print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)

---## 3. Load the dataset### 3.1 Where the data comes fromThe dataset is *Online Shoppers Purchasing Intention* from the UCI Machine Learning Repository. Each row isone **session** (one visit), not one visitor — a distinction that will matter repeatedly.The loader below tries a local copy first and falls back to the UCI URL. If both fail (no internet in thelab, for instance), download `online_shoppers_intention.csv` manually, place it beside this notebook, andre-run the cell.

In [ ]:
from pathlib import PathLOCAL_PATH = Path("online_shoppers_intention.csv")UCI_URL = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"           "00468/online_shoppers_intention.csv")def load_sessions():    '''Return the Online Shoppers dataframe, from a local file if present, else from UCI.'''    if LOCAL_PATH.exists():        print(f"Reading local file: {LOCAL_PATH.resolve()}")        return pd.read_csv(LOCAL_PATH)    try:        print("No local copy found — downloading from UCI ...")        frame = pd.read_csv(UCI_URL)        frame.to_csv(LOCAL_PATH, index=False)   # cache so the next run is offline        print(f"Downloaded and cached to {LOCAL_PATH.resolve()}")        return frame    except Exception as err:        raise FileNotFoundError(            "Could not load the dataset.\n"            "Download 'online_shoppers_intention.csv' from the UCI repository "            "(dataset 468, 'Online Shoppers Purchasing Intention Dataset'), "            "save it next to this notebook, and re-run this cell."        ) from errdf = load_sessions()print(f"\nLoaded {df.shape[0]:,} rows x {df.shape[1]} columns")

### 3.2 Verify before you analyseLoading is not the same as checking. Four questions, always, in this order: what shape is it, what types didpandas infer, what is missing, and are the rows unique?

In [ ]:
print("SHAPE")print(f"  rows    : {df.shape[0]:,}")print(f"  columns : {df.shape[1]}")print("\nDTYPES")print(df.dtypes.value_counts().to_string())print("\nMISSING VALUES")missing = df.isna().sum()if missing.sum() == 0:    print("  none reported by .isna() — but see the warning below")else:    print(missing[missing > 0].to_string())print("\nDUPLICATE ROWS")dups = df.duplicated().sum()print(f"  {dups:,} fully duplicated rows ({dups / len(df):.2%} of the file)")

> **Warning — a lesson from Lecture 1.** `.isna()` returning zero does **not** mean the data is complete. In> the Pima Diabetes dataset, 374 patients have an insulin level recorded as `0` — a physiologically impossible> value used as a placeholder, and completely invisible to a null check. Always look at the *minimum* of every> numeric column before you conclude that nothing is missing.Duplicated rows here are worth a moment's thought. Two sessions with identical values on all 18 columns areplausible — two short visits that bounced off the same page look the same in this schema. They are notautomatically errors, and deleting them would change the denominator of every rate we compute. We keep themand note the decision.

In [ ]:
# First look at the raw tabledf.head(5)

In [ ]:
# The minimum of every numeric column — the cheapest disguised-missing-value check there isnumeric_cols = df.select_dtypes(include=np.number).columns.tolist()mins = df[numeric_cols].min().rename("minimum").to_frame()mins["zeros"] = (df[numeric_cols] == 0).sum()mins["% zeros"] = (mins["zeros"] / len(df) * 100).round(1)mins.sort_values("% zeros", ascending=False)

**Interpretation.** High zero-counts here are *structural*, not errors. A session where the visitor neveropened an informational page genuinely has `Informational = 0` and `Informational_Duration = 0`. This is theopposite of the Pima case: there, zero was a lie standing in for a missing measurement; here, zero is thetruth. You can only tell the two apart by knowing what the column means — which is why a data dictionary ispart of EDA, not a formality that precedes it.

---## 4. Dependent and independent variables### 4.1 The dependent variable**`Revenue`** — a boolean: did this session end in a transaction?It is the dependent variable because it is the quantity whose behaviour we want to explain. Everything else inthe file is a candidate explanation. In modelling language it is the *target*; in experimental language it isthe *response*; in this notebook we mostly say *outcome*.### 4.2 The independent variablesThe other 17 columns are independent variables — but they are not all the same kind of thing, and the kinddetermines which statistics are legal. Grouping them properly now saves you from computing the mean of aregion code later.

In [ ]:
variable_roles = pd.DataFrame([    # column,                    role,          measurement type,          what it records    ("Revenue",                  "Dependent",   "Binary (bool)",           "Did the session end in a purchase?"),    ("Administrative",           "Independent", "Count (discrete)",        "Account/admin pages viewed"),    ("Administrative_Duration",  "Independent", "Continuous (seconds)",    "Time spent on admin pages"),    ("Informational",            "Independent", "Count (discrete)",        "Informational pages viewed"),    ("Informational_Duration",   "Independent", "Continuous (seconds)",    "Time spent on informational pages"),    ("ProductRelated",           "Independent", "Count (discrete)",        "Product pages viewed"),    ("ProductRelated_Duration",  "Independent", "Continuous (seconds)",    "Time spent on product pages"),    ("BounceRates",              "Independent", "Continuous (rate 0-1)",   "Avg. bounce rate of pages visited"),    ("ExitRates",                "Independent", "Continuous (rate 0-1)",   "Avg. exit rate of pages visited"),    ("PageValues",               "Independent", "Continuous (currency)",   "Avg. value of pages visited"),    ("SpecialDay",               "Independent", "Ordinal (0-1 scale)",     "Closeness to a special shopping day"),    ("Month",                    "Independent", "Nominal (ordered)",       "Month of the session"),    ("OperatingSystems",         "Independent", "Nominal (coded as int)",  "OS identifier"),    ("Browser",                  "Independent", "Nominal (coded as int)",  "Browser identifier"),    ("Region",                   "Independent", "Nominal (coded as int)",  "Geographic region identifier"),    ("TrafficType",              "Independent", "Nominal (coded as int)",  "Traffic source identifier"),    ("VisitorType",              "Independent", "Nominal (3 levels)",      "New / returning / other"),    ("Weekend",                  "Independent", "Binary (bool)",           "Did the session fall on a weekend?"),], columns=["column", "role", "measurement_type", "records"])variable_roles

### 4.3 The trap in this tableLook at `OperatingSystems`, `Browser`, `Region` and `TrafficType`. Pandas has read them as integers, so`df.mean()` will cheerfully return an average browser of 2.36. That number is meaningless: browser 4 is nottwice browser 2. These are **nominal categories that happen to be encoded as integers**, and treating them asnumeric is one of the most common silent errors in applied EDA.We therefore build an explicit list of the columns that are genuinely quantitative, and use it everywherebelow instead of `select_dtypes`.

In [ ]:
# Genuinely quantitative columns — the only ones for which mean, variance and skew are meaningfulQUANT = [    "Administrative", "Administrative_Duration",    "Informational", "Informational_Duration",    "ProductRelated", "ProductRelated_Duration",    "BounceRates", "ExitRates", "PageValues",]# Categorical columns, including the integer-coded onesCATEG = ["Month", "OperatingSystems", "Browser", "Region",         "TrafficType", "VisitorType", "Weekend", "SpecialDay"]TARGET = "Revenue"# The variable we profile in depth throughout this practicalFOCUS = "ProductRelated_Duration"print(f"{len(QUANT)} quantitative | {len(CATEG)} categorical | target = {TARGET}")print(f"Focus variable for the univariate work: {FOCUS}")

In [ ]:
# How balanced is the outcome? This single number reshapes the whole project.counts = df[TARGET].value_counts()rate = df[TARGET].mean()print("OUTCOME BALANCE")print(counts.to_string())print(f"\nConversion rate : {rate:.2%}")print(f"Majority class  : {(1 - rate):.2%} of sessions")print(f"\nA model that always predicts 'no purchase' would score {(1 - rate):.1%} accuracy")print("and would be completely useless. Accuracy is the wrong metric for this problem.")

---## 5. Mean vs. median### 5.1 What the comparison tells youThe mean is the balance point of the data; the median is the middle value. On a symmetric distribution theycoincide. The **direction and size of the gap between them is a diagnostic**:- mean ≈ median → roughly symmetric- mean > median → a long right tail is pulling the mean up- mean < median → a long left tail is pulling the mean downThe mean uses every value, so a single extreme observation moves it; the median depends only on rank, so itdoes not. That difference is what the *breakdown point* formalises: the mean has a breakdown point of 0%(one bad value is enough to move it anywhere), the median 50%.

In [ ]:
def centre_table(frame, cols):    '''Mean, median, trimmed mean and the mean-median gap for each column.'''    rows = []    for c in cols:        x = frame[c].dropna()        mean, median = x.mean(), x.median()        gap = mean - median        rel = (gap / median * 100) if median != 0 else np.nan        rows.append({            "variable": c,            "mean": mean,            "median": median,            "trimmed mean (10%)": stats.trim_mean(x, 0.1),            "mean - median": gap,            "gap as % of median": rel,        })    return pd.DataFrame(rows).set_index("variable")centre = centre_table(df, QUANT)centre

In [ ]:
# Read the table automatically, the way you would read it out loudprint("INTERPRETATION — mean vs. median\n" + "-" * 62)for c in QUANT:    mean, median = centre.loc[c, "mean"], centre.loc[c, "median"]    if median == 0:        verdict = ("median is exactly 0 — more than half of all sessions record nothing here, "                   "so the mean describes a minority of sessions")    else:        ratio = mean / median        if ratio > 1.25:            verdict = f"mean is {ratio:.2f}x the median -> clear right skew"        elif ratio < 0.80:            verdict = f"mean is {ratio:.2f}x the median -> left skew"        else:            verdict = f"mean and median within {abs(ratio - 1):.0%} -> roughly symmetric"    print(f"{c:<26} mean={mean:>12,.2f}  median={median:>10,.2f}   {verdict}")

### 5.2 The focus variable in detail`ProductRelated_Duration` is the total time, in seconds, a visitor spent on product pages during the session.It is the closest thing this dataset has to a measure of engagement, and it is the column we profile in depth.

In [ ]:
x = df[FOCUS].dropna()summary = pd.Series({    "count":              x.size,    "mean":               x.mean(),    "median":             x.median(),    "trimmed mean (10%)": stats.trim_mean(x, 0.1),    "minimum":            x.min(),    "25th percentile":    x.quantile(0.25),    "75th percentile":    x.quantile(0.75),    "90th percentile":    x.quantile(0.90),    "99th percentile":    x.quantile(0.99),    "maximum":            x.max(),})print(f"{FOCUS} (seconds)\n" + "-" * 46)print(summary.to_string(float_format=lambda v: f"{v:,.2f}"))

In [ ]:
mean_, med_ = x.mean(), x.median()share_below_mean = (x < mean_).mean()print("INTERPRETATION\n" + "-" * 62)print(f"The mean session spends {mean_:,.0f} seconds on product pages;")print(f"the median session spends {med_:,.0f} seconds — a factor of {mean_/med_:.2f}.")print(f"\n{share_below_mean:.1%} of all sessions fall BELOW the mean.")print("\nThat last number is the one to remember. When most of the population sits")print("below the average, the average has stopped describing a typical case. Report")print("the median as the headline figure, and quote the mean only alongside a note")print("about the skew.")

> **Reporting rule.** If you write "the average session lasts *m* seconds" in a report on this variable, you> are describing a session that most of your visitors never have. Say instead: *"a typical session spends> about X seconds on product pages; the distribution is strongly right-skewed, and a small number of very> long sessions pull the arithmetic mean up to Y."*

---## 6. Variance vs. IQR### 6.1 Two philosophies of spread| Measure | Definition | Uses | Breakdown point ||---|---|---|---|| Variance / SD | mean squared deviation from the mean | every value, squared | **0%** || IQR | Q3 − Q1 | the middle 50% only | **25%** || MAD | median of \|x − median\| | ranks only | **50%** |Squaring is what makes the variance fragile: a value ten times too large contributes one hundred times toomuch. The IQR cannot even see values outside the quartiles, and the MAD is more resistant still.A useful diagnostic is the ratio **SD / IQR**. Under a normal distribution the IQR is about 1.35 standarddeviations, so SD/IQR ≈ 0.74. Substantially above that means the tails are carrying weight the quartilesnever see.

In [ ]:
def spread_table(frame, cols):    rows = []    for c in cols:        v = frame[c].dropna()        q1, q3 = v.quantile(0.25), v.quantile(0.75)        iqr = q3 - q1        sd = v.std()        mad = (v - v.median()).abs().median()        rows.append({            "variable": c,            "variance": v.var(),            "std dev": sd,            "IQR": iqr,            "MAD": mad,            "SD / IQR": sd / iqr if iqr else np.nan,            "CV (SD/mean)": sd / v.mean() if v.mean() else np.nan,        })    return pd.DataFrame(rows).set_index("variable")spread = spread_table(df, QUANT)spread

In [ ]:
print("INTERPRETATION — variance vs. IQR\n" + "-" * 62)print("Reference: for a normal distribution, SD / IQR is about 0.74.\n")for c in QUANT:    ratio, iqr = spread.loc[c, "SD / IQR"], spread.loc[c, "IQR"]    if not np.isfinite(ratio):        note = "IQR is 0 — the middle half of the data is a single value"    elif ratio > 1.2:        note = f"SD/IQR = {ratio:.2f} -> far above normal; tails dominate the SD"    elif ratio > 0.9:        note = f"SD/IQR = {ratio:.2f} -> heavier tails than normal"    else:        note = f"SD/IQR = {ratio:.2f} -> close to normal behaviour"    print(f"{c:<26} IQR={iqr:>10,.2f}   {note}")

### 6.2 Demonstrating the breakdown pointLecture 2 claimed that the mean and standard deviation have a breakdown point of 0%, while the median and IQRsurvive up to 25–50% contamination. Here is that claim tested on the real column: we corrupt **one singlerow** — a plausible data-entry error where a duration in seconds is accidentally recorded in milliseconds —and recompute everything.

In [ ]:
contaminated = x.copy()worst_case = contaminated.max() * 1000          # one row, one mistyped unitcontaminated.iloc[0] = worst_casebefore = {"mean": x.mean(), "std dev": x.std(),          "median": x.median(), "IQR": x.quantile(.75) - x.quantile(.25)}after = {"mean": contaminated.mean(), "std dev": contaminated.std(),         "median": contaminated.median(),         "IQR": contaminated.quantile(.75) - contaminated.quantile(.25)}comparison = pd.DataFrame({"clean": before, "one bad row": after})comparison["change"] = ((comparison["one bad row"] - comparison["clean"])                        / comparison["clean"] * 100)comparison["change"] = comparison["change"].map(lambda v: f"{v:+.2f}%")print(f"One row of {len(x):,} changed to {worst_case:,.0f} seconds\n")print(comparison.to_string(float_format=lambda v: f"{v:,.2f}"))

In [ ]:
print("INTERPRETATION — breakdown point\n" + "-" * 62)print(f"One corrupted row out of {len(x):,} — that is {1/len(x):.4%} of the data — moved")print(f"the mean by {(after['mean']/before['mean'] - 1)*100:+.1f}% and the standard deviation by "      f"{(after['std dev']/before['std dev'] - 1)*100:+.1f}%.")print(f"The median moved by {(after['median']/before['median'] - 1)*100:+.1f}% and the IQR by "      f"{(after['IQR']/before['IQR'] - 1)*100:+.1f}%.")print("\nThis is not a quirk of this dataset. It is arithmetic: the mean is computed")print("from the values, the median from their ranks. Moving one point from the top")print("of the range to a thousand times the top of the range does not change its rank.")

---## 7. Skewness and distribution shape### 7.1 Definitions**Skewness** is the third standardised moment — a signed, unit-free measure of asymmetry. Positive means theright tail is longer.**Kurtosis** is the fourth standardised moment. `scipy.stats.kurtosis` returns *excess* kurtosis(kurtosis − 3), so a normal distribution scores 0. Positive excess means more probability in the tails than anormal curve predicts — i.e. extreme values happen more often than a 3-sigma rule would lead you to expect.A common working guide for the magnitude of skewness:| \|skew\| | Reading ||---|---|| < 0.5 | approximately symmetric || 0.5 – 1.0 | moderately skewed || > 1.0 | strongly skewed |

In [ ]:
def shape_table(frame, cols):    rows = []    for c in cols:        v = frame[c].dropna()        sk = stats.skew(v)        ku = stats.kurtosis(v)          # excess kurtosis: normal = 0        if abs(sk) < 0.5:            shape = "approximately symmetric"        elif abs(sk) < 1.0:            shape = "moderately " + ("right" if sk > 0 else "left") + "-skewed"        else:            shape = "strongly " + ("right" if sk > 0 else "left") + "-skewed"        rows.append({"variable": c, "skewness": sk, "excess kurtosis": ku,                     "shape": shape,                     "tails": "heavier than normal" if ku > 1 else                              ("lighter than normal" if ku < -0.5 else "near-normal")})    return pd.DataFrame(rows).set_index("variable")shape = shape_table(df, QUANT).sort_values("skewness", ascending=False)shape

In [ ]:
print("INTERPRETATION — shape\n" + "-" * 62)n_right = (shape["skewness"] > 1).sum()print(f"{n_right} of {len(QUANT)} quantitative columns are strongly right-skewed.\n")print("This is not an accident of this dataset. Every one of these variables is")print("bounded below by zero and unbounded above: a session cannot spend negative")print("time on a page, but it can spend an arbitrarily long time. Counts, durations,")print("prices, claim sizes and waiting times behave the same way for the same reason.")print("\nLeft skew, by contrast, needs a ceiling — exam scores near full marks, or")print("age at death in a developed country.")

### 7.2 What to do about skewSkew is not a defect to be removed on sight. It is information about the process that generated the data.But it does have four practical consequences, and a standard set of responses:1. **The mean stops describing anyone** → report the median.2. **Symmetric intervals become nonsense** → `mean ± 2·SD` can run below zero on a strictly positive quantity.3. **Squared-error models chase the tail** → consider a transform or a robust loss.4. **Distance-based methods distort** → k-means, kNN and PCA read the long tail as "very far away".The usual first response for strong right skew on non-negative data is `log1p`, which is `log(1 + x)` and istherefore safe at zero — important here, because many sessions record exactly 0 seconds.

In [ ]:
raw_skew = stats.skew(x)log_skew = stats.skew(np.log1p(x))sqrt_skew = stats.skew(np.sqrt(x))print(f"Skewness of {FOCUS}\n" + "-" * 46)print(f"  raw          : {raw_skew:>7.3f}")print(f"  sqrt(x)      : {sqrt_skew:>7.3f}")print(f"  log1p(x)     : {log_skew:>7.3f}")print(f"\nlog1p reduced the absolute skewness by "      f"{(1 - abs(log_skew)/abs(raw_skew)):.0%}.")print("\nNote what the transform costs you: the transformed variable is no longer in")print("seconds, so a coefficient fitted on it is no longer interpretable in seconds.")print("Transform for the model; report in the original units.")

---## 8. Histogram, box plot and densityThree views of the same column, because each one hides something the others show:- The **histogram** shows where the mass is, and any gaps, spikes or second peaks — but its appearance depends  on the bin width you chose.- The **box plot** shows the five-number summary and flags points beyond Tukey's fences — but it cannot show  multimodality, and a box drawn from 8 points looks identical to one drawn from 8,000.- The **density (KDE)** shows a smooth estimate of shape — but the smoothing is a modelling choice, and it will  happily place probability mass below zero on a strictly positive variable.Never use only one.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))# ---- top row: raw scaleaxes[0, 0].hist(x, bins=60, color=PRIMARY, edgecolor="white", linewidth=0.4)axes[0, 0].axvline(x.mean(), color="black", lw=2, ls="--", label=f"mean {x.mean():,.0f}")axes[0, 0].axvline(x.median(), color=SECONDARY, lw=2, label=f"median {x.median():,.0f}")axes[0, 0].set_title("Histogram (raw seconds)")axes[0, 0].set_xlabel(FOCUS); axes[0, 0].set_ylabel("sessions")axes[0, 0].legend(fontsize=8)axes[0, 1].boxplot(x, vert=False, widths=0.6,                   patch_artist=True,                   boxprops=dict(facecolor="#E9D9CE", edgecolor=PRIMARY, linewidth=1.5),                   medianprops=dict(color=PRIMARY, linewidth=2),                   flierprops=dict(marker="o", markersize=2.5,                                   markerfacecolor=PRIMARY, alpha=0.25,                                   markeredgecolor="none"))axes[0, 1].set_title("Box plot (raw seconds)")axes[0, 1].set_xlabel(FOCUS); axes[0, 1].set_yticks([])sns.kdeplot(x, ax=axes[0, 2], fill=True, color=PRIMARY, alpha=0.35, linewidth=2)axes[0, 2].set_title("Density (raw seconds)")axes[0, 2].set_xlabel(FOCUS)# ---- bottom row: log1p scalelx = np.log1p(x)axes[1, 0].hist(lx, bins=60, color=SECONDARY, edgecolor="white", linewidth=0.4)axes[1, 0].axvline(lx.mean(), color="black", lw=2, ls="--", label="mean")axes[1, 0].axvline(lx.median(), color=PRIMARY, lw=2, label="median")axes[1, 0].set_title("Histogram of log1p(x)")axes[1, 0].set_xlabel(f"log1p({FOCUS})"); axes[1, 0].set_ylabel("sessions")axes[1, 0].legend(fontsize=8)axes[1, 1].boxplot(lx, vert=False, widths=0.6,                   patch_artist=True,                   boxprops=dict(facecolor="#DDE6DF", edgecolor=SECONDARY, linewidth=1.5),                   medianprops=dict(color=SECONDARY, linewidth=2),                   flierprops=dict(marker="o", markersize=2.5,                                   markerfacecolor=SECONDARY, alpha=0.3,                                   markeredgecolor="none"))axes[1, 1].set_title("Box plot of log1p(x)")axes[1, 1].set_xlabel(f"log1p({FOCUS})"); axes[1, 1].set_yticks([])sns.kdeplot(lx, ax=axes[1, 2], fill=True, color=SECONDARY, alpha=0.35, linewidth=2)axes[1, 2].set_title("Density of log1p(x)")axes[1, 2].set_xlabel(f"log1p({FOCUS})")fig.suptitle(f"{FOCUS}: three views, two scales", fontsize=14, fontweight="bold")plt.show()

In [ ]:
q1, q3 = x.quantile(.25), x.quantile(.75)iqr = q3 - q1upper_fence = q3 + 1.5 * iqrflagged = (x > upper_fence).sum()print("INTERPRETATION — the three views\n" + "-" * 62)print(f"Tukey's upper fence: Q3 + 1.5 x IQR = {q3:,.1f} + 1.5 x {iqr:,.1f} = {upper_fence:,.1f} seconds")print(f"Sessions above the fence: {flagged:,} ({flagged/len(x):.2%} of the file)\n")print("The box plot flags these points. It does NOT call them errors. Recall the")print("three kinds of outlier from Lecture 2:")print("  1. data error        — would need a cause: a logging fault, a unit mix-up")print("  2. rare but real     — a genuine long browsing session")print("  3. different process — a bot, a scraper, a page left open overnight")print("\nThe number alone cannot distinguish them. Section 9 starts that investigation")print("by asking whether these long sessions behave differently on other variables.")print("\nNotice also what the log scale did: the raw histogram is an unreadable spike")print("against the y-axis, while the log1p histogram has visible shape. Both are the")print("same data. A plot is a claim about shape, and the scale is part of the claim.")

### 8.2 The same variable, split by outcomeUnivariate plots describe the column. The question we actually care about is comparative, so we redraw thesame three views split by `Revenue`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))palette = {False: MUTED, True: PRIMARY}plot_df = df.assign(**{"log_focus": np.log1p(df[FOCUS])})sns.histplot(data=plot_df, x="log_focus", hue=TARGET, bins=50, stat="density",             common_norm=False, element="step", fill=True, alpha=0.35,             palette=palette, ax=axes[0])axes[0].set_title("Histogram by outcome (density-normalised)")axes[0].set_xlabel(f"log1p({FOCUS})")sns.boxplot(data=plot_df, x=TARGET, y="log_focus", hue=TARGET,            palette=palette, legend=False, width=0.5, ax=axes[1])axes[1].set_title("Box plot by outcome")axes[1].set_xlabel("Purchase"); axes[1].set_ylabel(f"log1p({FOCUS})")sns.kdeplot(data=plot_df, x="log_focus", hue=TARGET, fill=True, alpha=0.3,            common_norm=False, palette=palette, linewidth=2, ax=axes[2])axes[2].set_title("Density by outcome")axes[2].set_xlabel(f"log1p({FOCUS})")fig.suptitle(f"{FOCUS} by purchase outcome", fontsize=14, fontweight="bold")plt.show()

In [ ]:
grp = df.groupby(TARGET)[FOCUS]buy, nobuy = grp.get_group(True), grp.get_group(False)print("INTERPRETATION — comparing the two groups\n" + "-" * 62)print(f"{'':<22}{'no purchase':>14}{'purchase':>14}")for label, fn in [("median (s)", np.median), ("mean (s)", np.mean)]:    print(f"{label:<22}{fn(nobuy):>14,.1f}{fn(buy):>14,.1f}")print(f"{'IQR (s)':<22}"      f"{nobuy.quantile(.75)-nobuy.quantile(.25):>14,.1f}"      f"{buy.quantile(.75)-buy.quantile(.25):>14,.1f}")print(f"{'n sessions':<22}{nobuy.size:>14,}{buy.size:>14,}")ratio_med = np.median(buy) / np.median(nobuy) if np.median(nobuy) else np.nanprint(f"\nPurchasing sessions have a median product-page time {ratio_med:.2f}x that of")print("non-purchasing sessions.")print("\nTwo cautions before you call this a finding:")print("  * This is association, not causation. Longer browsing may cause purchases,")print("    purchasing intent may cause longer browsing, or a third factor may cause both.")print("  * The groups are very unequal in size, so the purchase-group summary rests on")print("    far fewer sessions. Always print the group sizes next to the group statistics.")

---## 9. Pivot tablesA pivot table reshapes a long table into a comparison: rows are one grouping variable, columns another, andthe cells hold an aggregate. It is the fastest way to move from "what does this column look like" to "doesthis column behave differently across groups".Three rules that prevent most pivot-table mistakes:1. **Always print the group sizes.** A conversion rate of 100% from two sessions is not a finding.2. **Choose the aggregate to match the shape.** On skewed data use `median`, not `mean` — the whole of   Sections 5–7 was building to this.3. **Use `margins=True`.** The row and column totals are your sanity check against a filtering error.

In [ ]:
# Give Month a sensible order — the raw data stores it as an unordered stringMONTH_ORDER = ["Feb", "Mar", "Apr", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]present = [m for m in MONTH_ORDER if m in set(df["Month"])]df["Month"] = pd.Categorical(df["Month"], categories=present, ordered=True)print("Months present in the data:", ", ".join(present))

### 9.1 Conversion rate by month and visitor typeBecause `Revenue` is boolean, its **mean is the conversion rate** — a small trick worth remembering.

In [ ]:
conv = pd.pivot_table(df, values=TARGET, index="Month", columns="VisitorType",                      aggfunc="mean", observed=True)sizes = pd.pivot_table(df, values=TARGET, index="Month", columns="VisitorType",                       aggfunc="size", observed=True)print("CONVERSION RATE (%)")print((conv * 100).round(1).to_string(na_rep="-"))print("\nNUMBER OF SESSIONS BEHIND EACH CELL")print(sizes.to_string(na_rep="-"))

In [ ]:
print("INTERPRETATION — reading a pivot table safely\n" + "-" * 62)thin = sizes.stack().pipe(lambda s: s[s < 100])if len(thin):    print(f"{len(thin)} cells rest on fewer than 100 sessions:")    for (m, v), k in thin.items():        print(f"  {m:<6} x {v:<18} n = {k}")    print("\nRates in those cells are unstable — a handful of sessions either way moves")    print("them by several percentage points. Report them with the n, or not at all.")else:    print("Every cell rests on at least 100 sessions.")overall = df[TARGET].mean()print(f"\nOverall conversion rate for reference: {overall:.2%}")

### 9.2 Behaviour by outcome — mean and median side by sideThis is the pivot table that makes the argument of the whole practical. The same cells, aggregated two ways.

In [ ]:
by_mean = pd.pivot_table(df, values=QUANT, index=TARGET, aggfunc="mean").Tby_median = pd.pivot_table(df, values=QUANT, index=TARGET, aggfunc="median").Tcombined = pd.concat({"mean": by_mean, "median": by_median}, axis=1)combined

In [ ]:
print("INTERPRETATION — why the aggregate you choose changes the story\n" + "-" * 68)print(f"{'variable':<26}{'mean ratio':>12}{'median ratio':>14}")print("-" * 68)for c in QUANT:    m0, m1 = by_mean.loc[c, False], by_mean.loc[c, True]    d0, d1 = by_median.loc[c, False], by_median.loc[c, True]    mr = m1 / m0 if m0 else np.nan    dr = d1 / d0 if d0 else np.nan    print(f"{c:<26}{mr:>12.2f}{dr:>14.2f}" if np.isfinite(dr)          else f"{c:<26}{mr:>12.2f}{'n/a':>14}")print("\nEach number is the purchase-group value divided by the no-purchase-group value.")print("Where the two ratios disagree, the mean ratio is being driven by extreme")print("sessions rather than by typical ones — and the median ratio is the more")print("honest summary of how a typical purchasing session differs.")print("\n'n/a' means the no-purchase median is exactly 0, so a ratio is undefined.")print("That is itself a finding: more than half of non-purchasing sessions record")print("nothing at all on that variable.")

### 9.3 A robust pivot: median engagement by month and outcome

In [ ]:
robust_pivot = pd.pivot_table(df, values=FOCUS, index="Month", columns=TARGET,                              aggfunc="median", margins=True, margins_name="All",                              observed=True)robust_pivot.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))heat = pd.pivot_table(df, values=TARGET, index="Month", columns="VisitorType",                      aggfunc="mean", observed=True) * 100sns.heatmap(heat, annot=True, fmt=".1f", cmap="YlOrBr", linewidths=0.5,            linecolor="white", cbar_kws={"label": "conversion rate (%)"}, ax=ax)ax.set_title("Conversion rate (%) by month and visitor type", fontweight="bold")ax.set_xlabel("Visitor type"); ax.set_ylabel("Month")plt.show()

### 9.4 Two-way categorical pivot: weekend × visitor type

In [ ]:
weekend_pivot = pd.pivot_table(df, values=TARGET, index="Weekend",                               columns="VisitorType", aggfunc=["mean", "size"],                               observed=True)print("Conversion rate and session count by weekend and visitor type")print(weekend_pivot.to_string())print("\n\nCross-tabulation of Weekend x Revenue (row percentages)")print((pd.crosstab(df["Weekend"], df[TARGET], normalize="index") * 100).round(2).to_string())

In [ ]:
wk = df.groupby("Weekend", observed=True)[TARGET].agg(["mean", "size"])diff = (wk.loc[True, "mean"] - wk.loc[False, "mean"]) * 100print("INTERPRETATION — weekend effect\n" + "-" * 62)print(f"Weekday conversion : {wk.loc[False, 'mean']:.2%}  (n = {wk.loc[False, 'size']:,})")print(f"Weekend conversion : {wk.loc[True,  'mean']:.2%}  (n = {wk.loc[True,  'size']:,})")print(f"Difference         : {diff:+.2f} percentage points")print("\nThis is an exploratory observation, not a tested claim. Lecture 1's rule")print("applies: a difference found by looking is a hypothesis. To claim it as a")print("finding you would state the comparison in advance and test it on data you")print("had not explored — and you would check whether the weekend and weekday")print("groups differ in month composition, since the months differ enormously in")print("conversion and could produce this gap on their own.")

---## 10. Findings logEvery practical in DATA209 ends with a findings log. It is one page, written in plain sentences, and it is theartefact that survives after the notebook is closed. Assignment 1 is, in effect, a long findings log.Fill this in from your own run — the numbers your code produced, not the ones in any example.### 10.1 Structure and quality| Item | What you found | Decision taken ||---|---|---|| Rows × columns | | || Missing values | | || Duplicated rows | | || Disguised missing values (zeros) | | || Integer-coded categoricals | | |### 10.2 Distribution shape| Variable | Median | Mean | Skewness | Shape | Statistic you will report ||---|---|---|---|---|---|| ProductRelated_Duration | | | | | || PageValues | | | | | || BounceRates | | | | | |### 10.3 Claims, and their status| # | Claim | Evidence | Exploratory or confirmatory? ||---|---|---|---|| 1 | | | Exploratory || 2 | | | Exploratory || 3 | | | Exploratory |> Every claim produced in this practical is **exploratory**. Nothing here was pre-specified, and several> variables were examined before choosing what to report. That is exactly what EDA is for — and exactly why> none of it may be reported with confirmatory language.### 10.4 Open questions for P3–41.2.3.

---## 11. ExercisesWork through these in this notebook. Each one is short; the marks are in the interpretation, not the code.**E1 — Repeat the profile.** Choose `PageValues` instead of `ProductRelated_Duration` as the focus variable.Compute the mean, median, variance, IQR and skewness, and produce the three-panel plot. `PageValues` has alarge spike at exactly zero — explain what that does to the median, and whether the mean or the median is themore useful summary here.**E2 — Fences and judgement.** Using Tukey's rule, count the flagged sessions in `Administrative_Duration`.Inspect ten of those rows in full. For each, argue whether it is a data error, rare-but-real, or a differentprocess. State which of the three the *majority* appear to be, and what you would do about them.**E3 — Robustness in the group comparison.** Recompute Section 8.2's group comparison using the trimmed mean(10%) instead of the mean. Does the conclusion about purchasing sessions change? Explain in two sentences whyit did or did not.**E4 — A pivot table of your own.** Build a pivot table of conversion rate by `TrafficType` (rows) and`VisitorType` (columns), with session counts alongside. Identify the two cells you would refuse to report, andsay why.**E5 — Simpson's paradox check.** The weekend effect in Section 9.4 might be produced by month composition.Compute the weekend-vs-weekday conversion difference *within each month*, and compare the pattern to thepooled figure. Does the pooled difference hold up inside every month?**E6 — Write the log.** Complete Section 10 in full sentences. A marker who has never seen this dataset shouldbe able to read your log alone and know what the data looks like and what you would do next.

---### Deliverables for P1–21. This notebook, run top to bottom with all outputs visible.2. Sections 10 and 11 completed.3. Committed to your group's GitHub repository, with a one-line commit message describing what you found.### Looking ahead**P3–4** takes the flagged sessions from Section 8 and treats them as the subject rather than a nuisance:outlier detection rules, their assumptions, and the masking effect. The findings log you write today is theinput to that practical.